## Orchestrator 테스트

<div style="text-align: right"> <b>KMYU</b></div>
<div style="text-align: right"> Initial issue : 2026.04.02 </div>
<div style="text-align: right"> last update : 2026.04.02 </div>

개정 이력  
- `2026.04.02` : 노트북 최초 작성 — ReAct Orchestrator E2E 테스트 (구 04_graph_test.ipynb 대체)

### 아키텍처 개요

현재 에이전트는 **3계층 ReAct 구조**로 동작합니다:

```
Orchestrator (ReAct Agent + 7 META_TOOLS)
    ↓ run_deal_structuring(deal_id)
    ↓ [HOLD 판정 시] → run_final_verdict(deal_id, hold=true) → END
    ↓ [정상] 4개 Worker 병렬 호출:
        ├─ run_scoring_analysis(deal_id)
        ├─ run_resource_estimation(deal_id)
        ├─ run_risk_analysis(deal_id)
        └─ run_similar_project_search(deal_id)
    ↓ run_final_verdict(deal_id)
    ↓ END
```

**구버전과의 핵심 차이:**
- 구: 정적 LangGraph StateGraph (고정된 edge/conditional)
- 신: ReAct Orchestrator LLM이 실행 순서를 동적으로 결정
- State: `AnalysisContext` 레지스트리 패턴 (모듈 레벨 dict)
- 입력: `{"messages": [HumanMessage(...)]}` (구: `{"deal_id": ..., "deal_input": ...}`)
- 결과: `cleanup_analysis_context(deal_id)` → result_dict (구: graph return value)

**사전 요구사항:** `make docker-up && make migrate && make seed` + Pinecone 설정 + `.env` API keys

In [ ]:
import json
import sys
import uuid
import time

from dotenv import load_dotenv

load_dotenv()
sys.path.insert(0, "../../")

---
### 1. AnalysisContext 설정

In [ ]:
from backend.app.agent.orchestrator.context import (
    cleanup_analysis_context,
    get_analysis_context,
    init_analysis_context,
)

# AnalysisContext 라이프사이클 확인
test_ctx_id = str(uuid.uuid4())
ctx = init_analysis_context(deal_id=test_ctx_id, deal_input="test")
print(f"✓ AnalysisContext 생성: deal_id={test_ctx_id}")
print(f"  to_agent_state keys: {list(ctx.to_agent_state().keys())}")
print(f"  to_result_dict keys: {list(ctx.to_result_dict().keys())}")

# 정리
cleanup_result = cleanup_analysis_context(test_ctx_id)
print(f"\n✓ cleanup 완료: {cleanup_result is not None}")

---
### 2. Orchestrator 빌드

In [ ]:
from backend.app.agent.graph import build_graph
from backend.app.agent.orchestrator.agent import ORCHESTRATOR_MAX_ITERATIONS

deal_id = str(uuid.uuid4())
graph = build_graph(deal_id=deal_id)

print(f"✓ Orchestrator 빌드 완료")
print(f"  type: {type(graph).__name__}")
print(f"  max_iterations: {ORCHESTRATOR_MAX_ITERATIONS}")
print(f"  recursion_limit: {2 * ORCHESTRATOR_MAX_ITERATIONS + 1}")

---
### 3. 정상 딜 E2E 실행

In [ ]:
sample_deal_input = """[딜 기본 정보]
고객사: 메가테크
고객 규모: 대기업
산업군: 제조업
프로젝트명: 공장 설비 예지보전 AI 시스템 개발
예상 금액: 20000만원
금액 단위: 만원
기간: 5개월

[상세 내용]
## 프로젝트 배경
기존 정기 점검 체계에서 AI 기반 예측 유지보수로 전환하여 설비 가동률 향상 및 유지보수 비용 절감을 목표로 함.
현재 설비 고장으로 인한 비계획 정지가 연간 약 50회 발생하며, 이를 80% 이상 감소시키는 것이 핵심 목표.

## 기술 요구사항
- Python, TensorFlow 기반 시계열 예측 모델
- IoT 센서 데이터 실시간 수집/처리 파이프라인
- 대시보드 (React 기반)
- MLOps: 모델 재학습 자동화 파이프라인

## 데이터 요구사항
- 3년간 설비 센서 데이터 (진동, 온도, 전류) 약 500GB
- 설비 고장 이력 데이터

## 보안 요구사항
- 온프레미스 배포 필수

## 결제 조건
착수금 30%, 중간 40%, 완료 30%

## 이해관계자
- 발주측 PM: 설비관리팀 김부장
- 의사결정권자: 제조본부장"""

print(f"deal_id: {deal_id}")
print(f"deal_input 길이: {len(sample_deal_input)} chars")

In [ ]:
# AnalysisContext 초기화
ctx = init_analysis_context(deal_id=deal_id, deal_input=sample_deal_input)

# 사용자 메시지 구성
user_message = f"""다음 딜을 분석해주세요.

deal_id: {deal_id}

{sample_deal_input}"""

print(f"✓ AnalysisContext 초기화 완료")
print(f"  deal_id: {deal_id}")

In [ ]:
from langchain_core.messages import HumanMessage

# E2E 실행
start_time = time.time()

result = await graph.ainvoke(
    {"messages": [HumanMessage(content=user_message)]},
    config={
        "recursion_limit": 2 * ORCHESTRATOR_MAX_ITERATIONS + 1,
    },
)

elapsed = time.time() - start_time
print(f"✓ E2E 실행 완료: {elapsed:.1f}초")

In [ ]:
# AnalysisContext에서 결과 추출
analysis_result = cleanup_analysis_context(deal_id)

if analysis_result:
    print("=== 분석 결과 요약 ===")
    print(f"structured_deal: {'있음' if analysis_result.get('structured_deal') else '없음'}")
    print(f"scores: {len(analysis_result.get('scores', []))}개 기준")
    print(f"total_score: {analysis_result.get('total_score', 'N/A')}")
    print(f"verdict: {analysis_result.get('verdict', 'N/A')}")
    print(f"resource_estimate: {'있음' if analysis_result.get('resource_estimate') else '없음'}")
    print(f"risks: {len(analysis_result.get('risks', []))}개")
    print(f"risk_interdependencies: {len(analysis_result.get('risk_interdependencies', []))}개")
    print(f"similar_projects: {len(analysis_result.get('similar_projects', []))}개")
    print(f"final_report: {len(analysis_result.get('final_report', ''))} chars")
    print(f"errors: {analysis_result.get('errors', [])}")
else:
    print("✗ AnalysisContext에서 결과를 찾을 수 없음")

---
### 4. 결과 상세 분석

In [ ]:
# Structured Deal
if analysis_result:
    sd = analysis_result.get("structured_deal", {})
    print("=== Structured Deal ===")
    print(f"customer_name: {sd.get('customer_name')}")
    print(f"customer_industry: {sd.get('customer_industry')}")
    print(f"expected_amount: {sd.get('expected_amount')}만원")
    print(f"duration_months: {sd.get('duration_months')}")
    print(f"missing_fields: {sd.get('missing_fields', [])}")

In [ ]:
# Scoring
if analysis_result:
    print("=== Scoring ===")
    for s in analysis_result.get("scores", []):
        print(f"  {s.get('criterion')}: {s.get('score')}점 × {s.get('weight')} = {s.get('weighted_score')}")
    print(f"\n총점: {analysis_result.get('total_score')} → {analysis_result.get('verdict')}")

In [ ]:
# Resource Estimation
if analysis_result and analysis_result.get("resource_estimate"):
    re = analysis_result["resource_estimate"]
    print("=== Resource Estimation ===")
    if "profitability" in re:
        prof = re["profitability"]
        print(f"  deal_amount: {prof.get('deal_amount')}만원")
        print(f"  estimated_cost: {prof.get('estimated_cost')}만원")
        print(f"  expected_margin: {prof.get('expected_margin')}")
    if "duration_months" in re:
        print(f"  duration: {re['duration_months']}개월 (with buffer: {re.get('duration_with_buffer')})")

In [ ]:
# Risks
if analysis_result:
    print("=== Risk Analysis ===")
    for r in analysis_result.get("risks", []):
        print(f"  [{r.get('level', 'N/A')}] {r.get('category', '')} — {r.get('item', '')}")
    interdeps = analysis_result.get("risk_interdependencies", [])
    if interdeps:
        print(f"\n리스크 상호작용 ({len(interdeps)}건):")
        for ri in interdeps:
            print(f"  {ri.get('risk_pair', [])} → {ri.get('combined_effect', '')[:80]}")

In [ ]:
# Final Report 마크다운 렌더링
from IPython.display import Markdown, display

if analysis_result and analysis_result.get("final_report"):
    report = analysis_result["final_report"]
    
    # 8개 섹션 존재 확인
    expected_sections = ["핵심 결론", "Deal 개요", "Deal 선정 기준 적합성", "평가 상세",
                         "예상 작업 범위", "리스크 분석", "유사 프로젝트", "권고사항"]
    for section in expected_sections:
        status = "✓" if section in report else "✗"
        print(f"  {status} '{section}' 섹션")
    
    print("\n" + "="*60)
    display(Markdown(report))

---
### 5. Hold 케이스 E2E

critical field 3개 이상 누락 시 Orchestrator가 Hold 프로토콜을 따르는지 확인합니다.  
deal_structuring → HOLD_RECOMMENDED → final_verdict(hold=true)

In [ ]:
hold_deal_id = str(uuid.uuid4())
hold_deal_input = """[딜 기본 정보]
프로젝트명: AI 프로젝트

[상세 내용]
AI를 활용한 프로젝트를 진행하고 싶습니다. 상세 내용은 추후 공유 예정입니다."""

# AnalysisContext 초기화
hold_ctx = init_analysis_context(deal_id=hold_deal_id, deal_input=hold_deal_input)

# Orchestrator 빌드 및 실행
hold_graph = build_graph(deal_id=hold_deal_id)

hold_message = f"""다음 딜을 분석해주세요.

deal_id: {hold_deal_id}

{hold_deal_input}"""

start_time = time.time()
hold_result = await hold_graph.ainvoke(
    {"messages": [HumanMessage(content=hold_message)]},
    config={"recursion_limit": 2 * ORCHESTRATOR_MAX_ITERATIONS + 1},
)
elapsed = time.time() - start_time
print(f"✓ Hold 케이스 실행 완료: {elapsed:.1f}초")

In [ ]:
hold_analysis = cleanup_analysis_context(hold_deal_id)

if hold_analysis:
    print("=== Hold 케이스 결과 ===")
    print(f"verdict: {hold_analysis.get('verdict')}")
    print(f"total_score: {hold_analysis.get('total_score')}")
    print(f"missing_fields: {hold_analysis.get('structured_deal', {}).get('missing_fields', [])}")
    print(f"final_report 길이: {len(hold_analysis.get('final_report', ''))} chars")
    print(f"errors: {hold_analysis.get('errors', [])}")
    
    # Hold 검증
    if hold_analysis.get('verdict') == 'pending':
        print("\n✓ Hold 프로토콜 정상 동작")
    else:
        print(f"\n⚠ verdict가 'pending'이 아님: {hold_analysis.get('verdict')}")
else:
    print("✗ Hold 케이스 결과 없음")

---
### 6. 스트리밍 실행

Orchestrator의 reasoning step과 meta-tool 호출을 실시간으로 추적합니다.  
ReAct agent의 message-based 스트리밍 패턴을 사용합니다.

In [ ]:
stream_deal_id = str(uuid.uuid4())
stream_ctx = init_analysis_context(deal_id=stream_deal_id, deal_input=sample_deal_input)
stream_graph = build_graph(deal_id=stream_deal_id)

stream_message = f"""다음 딜을 분석해주세요.

deal_id: {stream_deal_id}

{sample_deal_input}"""

print("=== 스트리밍 실행 시작 ===")
start_time = time.time()
step_count = 0

async for event in stream_graph.astream(
    {"messages": [HumanMessage(content=stream_message)]},
    config={"recursion_limit": 2 * ORCHESTRATOR_MAX_ITERATIONS + 1},
    stream_mode="updates",
):
    step_count += 1
    elapsed = time.time() - start_time
    
    for node_name, update in event.items():
        messages = update.get("messages", [])
        for msg in messages:
            msg_type = type(msg).__name__
            content_preview = ""
            if hasattr(msg, "content") and msg.content:
                content_preview = str(msg.content)[:100]
            elif hasattr(msg, "tool_calls") and msg.tool_calls:
                tool_names = [tc["name"] for tc in msg.tool_calls]
                content_preview = f"tools: {tool_names}"
            
            print(f"[{elapsed:6.1f}s] step={step_count} node={node_name} type={msg_type}: {content_preview}")

total_elapsed = time.time() - start_time
print(f"\n=== 스트리밍 완료: {total_elapsed:.1f}초, {step_count} steps ===")

# 결과 정리
stream_result = cleanup_analysis_context(stream_deal_id)
if stream_result:
    print(f"verdict: {stream_result.get('verdict')}, total_score: {stream_result.get('total_score')}")

---
### 7. 계층적 로깅 검증

OrchestratorLogCallback을 설정하여 실행 후 AgentLog 테이블에서 계층 구조를 확인합니다.

```
orchestrator_reasoning (LLM 사고)
  → orchestrator_tool_call (meta-tool 호출)
    → worker_start (Worker 시작, parent_log_id로 연결)
      → reasoning / tool_call / observation (Worker 내부 step)
```

In [ ]:
from backend.app.agent.orchestrator.callback import OrchestratorLogCallback

log_deal_id = str(uuid.uuid4())
log_deal_uuid = uuid.UUID(log_deal_id)

# AnalysisContext + OrchestratorLogCallback 설정
log_ctx = init_analysis_context(deal_id=log_deal_id, deal_input=sample_deal_input)
orch_callback = OrchestratorLogCallback(deal_id=log_deal_uuid)
log_ctx.orchestrator_callback = orch_callback

log_graph = build_graph(deal_id=log_deal_id)

log_message = f"""다음 딜을 분석해주세요.

deal_id: {log_deal_id}

{sample_deal_input}"""

print("=== 로깅 포함 실행 시작 ===")
start_time = time.time()

await log_graph.ainvoke(
    {"messages": [HumanMessage(content=log_message)]},
    config={
        "recursion_limit": 2 * ORCHESTRATOR_MAX_ITERATIONS + 1,
        "callbacks": [orch_callback],
    },
)

elapsed = time.time() - start_time
print(f"✓ 실행 완료: {elapsed:.1f}초")

log_result = cleanup_analysis_context(log_deal_id)
print(f"verdict: {log_result.get('verdict') if log_result else 'N/A'}")

In [ ]:
# AgentLog 테이블에서 로그 계층 구조 확인
from backend.app.db.repositories import agent_log_repo
from backend.app.db.session import AsyncSessionLocal

async with AsyncSessionLocal() as session:
    logs = await agent_log_repo.list_by_deal_id(session, log_deal_uuid)

if logs:
    print(f"=== AgentLog 레코드 수: {len(logs)} ===")
    
    # step_type별 분류
    by_type = {}
    for log in logs:
        st = log.step_type or "unknown"
        by_type.setdefault(st, []).append(log)
    
    print(f"\nStep type 분포:")
    for st, entries in sorted(by_type.items()):
        print(f"  {st}: {len(entries)}건")
    
    # 계층 구조 확인 (parent_log_id 관계)
    print(f"\n계층 구조 (parent_log_id 연결):")
    parent_ids = {log.parent_log_id for log in logs if log.parent_log_id}
    print(f"  parent_log_id가 있는 로그: {len([l for l in logs if l.parent_log_id])}건")
    print(f"  고유 parent_id 수: {len(parent_ids)}개")
    
    # 상위 5개 로그 미리보기
    print(f"\n최근 로그 (시간순):")
    for log in sorted(logs, key=lambda l: l.started_at or l.created_at)[:10]:
        parent = f" (parent: {str(log.parent_log_id)[:8]}...)" if log.parent_log_id else ""
        output_preview = (log.raw_output or "")[:60]
        print(f"  [{log.step_type}] {log.node_name} — {log.duration_ms}ms{parent}")
        if output_preview:
            print(f"    output: {output_preview}...")
else:
    print("✗ AgentLog 레코드 없음")

---
### 8. Orchestrator 메시지 히스토리 분석

Orchestrator의 실제 reasoning 과정과 tool 호출 순서를 확인합니다.

In [ ]:
# ainvoke 결과에서 메시지 히스토리 확인
if result:
    messages = result.get("messages", [])
    print(f"=== Orchestrator 메시지 히스토리 ({len(messages)}개) ===")
    for i, msg in enumerate(messages):
        msg_type = type(msg).__name__
        
        if msg_type == "HumanMessage":
            print(f"\n[{i}] {msg_type}: {str(msg.content)[:100]}...")
        elif msg_type == "AIMessage":
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                tool_names = [tc["name"] for tc in msg.tool_calls]
                print(f"\n[{i}] {msg_type} (tool_calls: {tool_names})")
                if msg.content:
                    print(f"    reasoning: {str(msg.content)[:200]}")
            else:
                print(f"\n[{i}] {msg_type}: {str(msg.content)[:200]}...")
        elif msg_type == "ToolMessage":
            print(f"[{i}] {msg_type} (tool={msg.name}): {str(msg.content)[:100]}...")
        else:
            print(f"\n[{i}] {msg_type}: {str(msg)[:100]}...")